# Fase 2 — `Dataset` y `DataLoader` en PyTorch

**Objetivo:** convertir los GeoTIFF en tensores listos para la U-Net, de forma correcta y verificable.

| Pieza | Qué hace | Por qué |
|---|---|---|
| `Normalizador` | recorta a [p1, p99] y estandariza con media/desv. de **train** | entradas centradas en 0 → gradientes estables; sin fuga de información |
| NaN → 0 y etiqueta → **-1** | los 2.5 M píxeles SAR no finitos se neutralizan | un solo NaN arruina todos los gradientes; y no tiene sentido aprender/evaluar donde no hay señal |
| Etiqueta **-1 = ignorar** | la pérdida y las métricas saltan esos píxeles | sin dato (nubes en S2) + SAR inválido |
| Aumento (solo train) | volteos + rotaciones 90°, **iguales** en imagen y máscara | 252 chips es poco: multiplica la variedad sin inventar píxeles |
| Recorte aleatorio 256×256 (solo train) | cada época ve una ventana distinta de cada chip | más variedad, lotes más grandes en la T4; la U-Net es totalmente convolucional, así que luego predice en 512×512 sin problema |
| Caché en RAM | los chips a mano se leen del disco una sola vez | ~0.9 GB, cabe de sobra; acelera cada época |

## 0. Preparación

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
RAIZ_PROYECTO = Path('/content/drive/MyDrive/rnap_p2_inundaciones')
sys.path.insert(0, str(RAIZ_PROYECTO))

!pip install -q rasterio

## 1. Módulos nuevos de `src/`
Solo archivos **nuevos**; no se toca nada de las fases anteriores.

In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/preprocesamiento.py
"""
preprocesamiento.py
Normalización de las bandas SAR (VV, VH) con las estadísticas calculadas SOLO en train (fase 1).
Pasos:  1) detectar píxeles no finitos (NaN/inf)
        2) recortar a [p1, p99]  -> evita que valores extremos dominen
        3) estandarizar (x - media) / desviacion  -> entradas centradas en 0 y escala ~1
        4) NaN -> 0 (que tras estandarizar = "valor medio": la red no recibe basura)
"""
import json

import numpy as np

BANDAS = ["vv", "vh"]


class Normalizador:
    def __init__(self, estadisticas: dict):
        self.minimos = np.array([estadisticas[b]["p1"] for b in BANDAS], dtype=np.float32)[:, None, None]
        self.maximos = np.array([estadisticas[b]["p99"] for b in BANDAS], dtype=np.float32)[:, None, None]
        self.medias = np.array([estadisticas[b]["media"] for b in BANDAS], dtype=np.float32)[:, None, None]
        self.desviaciones = np.array([estadisticas[b]["desviacion"] for b in BANDAS], dtype=np.float32)[:, None, None]

    @classmethod
    def desde_json(cls, ruta) -> "Normalizador":
        with open(ruta) as f:
            return cls(json.load(f))

    def __call__(self, s1: np.ndarray):
        """s1: (2, H, W) en dB -> (s1 normalizado (2, H, W), máscara de píxeles finitos (H, W))."""
        finitos = np.isfinite(s1).all(axis=0)
        x = np.clip(s1, self.minimos, self.maximos)
        x = (x - self.medias) / self.desviaciones
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        return x.astype(np.float32), finitos

    def revertir(self, x: np.ndarray) -> np.ndarray:
        """De vuelta a dB (útil para visualizar lo que entra a la red)."""
        return x * self.desviaciones + self.medias


In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/aumentos.py
"""
aumentos.py
Aumento de datos geométrico. Regla de oro en segmentación: la MISMA transformación
a la imagen y a la máscara, si no la etiqueta deja de corresponder al píxel.

Solo usamos volteos y rotaciones de 90°: no inventan valores (no hay interpolación),
así que la máscara sigue siendo exactamente {-1, 0, 1}.
Matiz para el informe: el SAR mira de lado (geometría oblicua), por lo que la dirección
de sombras/layover no es invariante a rotaciones; aun así es la práctica estándar
y mejora la generalización con pocos datos (446 chips).
"""
import numpy as np


def aumentar(imagen: np.ndarray, etiqueta: np.ndarray, generador: np.random.Generator):
    """imagen: (C, H, W); etiqueta: (H, W)."""
    if generador.random() < 0.5:                       # volteo horizontal
        imagen, etiqueta = imagen[:, :, ::-1], etiqueta[:, ::-1]
    if generador.random() < 0.5:                       # volteo vertical
        imagen, etiqueta = imagen[:, ::-1, :], etiqueta[::-1, :]
    k = int(generador.integers(0, 4))                  # rotación 0/90/180/270
    if k:
        imagen = np.rot90(imagen, k, axes=(1, 2))
        etiqueta = np.rot90(etiqueta, k, axes=(0, 1))
    return np.ascontiguousarray(imagen), np.ascontiguousarray(etiqueta)


def recorte_aleatorio(imagen: np.ndarray, etiqueta: np.ndarray, tamano: int, generador: np.random.Generator):
    """Recorta una ventana tamano x tamano en posición aleatoria (otra forma de aumento)."""
    _, alto, ancho = imagen.shape
    fila = int(generador.integers(0, alto - tamano + 1))
    columna = int(generador.integers(0, ancho - tamano + 1))
    return (imagen[:, fila:fila + tamano, columna:columna + tamano],
            etiqueta[fila:fila + tamano, columna:columna + tamano])


In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/conjunto_datos.py
"""
conjunto_datos.py
Dataset de PyTorch para Sen1Floods11.
Devuelve (imagen, etiqueta, id_chip):
  imagen   : tensor float32 (2, H, W)  -> VV, VH normalizados
  etiqueta : tensor int64   (H, W)     -> 0 no agua, 1 agua, -1 IGNORAR (sin dato o SAR no finito)
"""
import numpy as np
import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm

from .aumentos import aumentar, recorte_aleatorio
from .configuracion import VALOR_SIN_DATO, DIR_S1_MANO, DIR_ETIQUETA_MANO
from .lectura import leer_s1, leer_etiqueta


class ConjuntoInundaciones(Dataset):
    def __init__(self, ids_chips, normalizador, aumentar_datos: bool = False,
                 tamano_recorte: int | None = None, en_memoria: bool = True,
                 dir_s1=DIR_S1_MANO, sufijo_s1: str = "S1Hand",
                 dir_etiqueta=DIR_ETIQUETA_MANO, sufijo_etiqueta: str = "LabelHand",
                 semilla: int = 42):
        self.ids_chips = list(ids_chips)
        self.normalizador = normalizador
        self.aumentar_datos = aumentar_datos
        self.tamano_recorte = tamano_recorte
        self.dir_s1, self.sufijo_s1 = dir_s1, sufijo_s1
        self.dir_etiqueta, self.sufijo_etiqueta = dir_etiqueta, sufijo_etiqueta
        self.generador = np.random.default_rng(semilla)

        # Los chips a mano caben en RAM (~2 MB c/u): se cargan UNA vez, antes de que los
        # workers del DataLoader se creen (así lo comparten en vez de leer disco cada época).
        self.cache = None
        if en_memoria:
            self.cache = [self._cargar(i) for i in tqdm(self.ids_chips, desc="Cargando chips en RAM")]

    def _cargar(self, id_chip):
        s1 = leer_s1(id_chip, self.dir_s1, self.sufijo_s1)
        etiqueta = leer_etiqueta(id_chip, self.dir_etiqueta, self.sufijo_etiqueta)
        imagen, finitos = self.normalizador(s1)
        etiqueta = etiqueta.copy()
        etiqueta[~finitos] = VALOR_SIN_DATO          # SAR inválido -> no se evalúa ni se aprende
        return imagen, etiqueta.astype(np.int64)

    def __len__(self):
        return len(self.ids_chips)

    def __getitem__(self, indice):
        imagen, etiqueta = self.cache[indice] if self.cache is not None else self._cargar(self.ids_chips[indice])
        if self.tamano_recorte:
            imagen, etiqueta = recorte_aleatorio(imagen, etiqueta, self.tamano_recorte, self.generador)
        if self.aumentar_datos:
            imagen, etiqueta = aumentar(imagen, etiqueta, self.generador)
        return torch.from_numpy(np.ascontiguousarray(imagen)), \
               torch.from_numpy(np.ascontiguousarray(etiqueta)), self.ids_chips[indice]


def semilla_por_worker(id_worker: int):
    """Cada worker del DataLoader recibe su propio generador (si no, todos aumentarían igual)."""
    informacion = torch.utils.data.get_worker_info()
    semilla = (torch.initial_seed() + id_worker) % 2**32
    informacion.dataset.generador = np.random.default_rng(semilla)


In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/cargadores.py
"""
cargadores.py
Crea los DataLoader de las 4 particiones oficiales de los chips a mano.
Solo 'train' lleva aumento de datos (y recorte opcional); valid/test/bolivia se evalúan tal cual.
"""
import torch
from torch.utils.data import DataLoader

from .conjunto_datos import ConjuntoInundaciones, semilla_por_worker
from .lectura import leer_particion


def crear_cargadores(normalizador, tamano_lote: int = 8, tamano_recorte: int | None = None,
                     num_workers: int = 2, semilla: int = 42) -> dict:
    cargadores = {}
    for particion in ["train", "valid", "test", "bolivia"]:
        es_train = particion == "train"
        ids = leer_particion(particion)["id_chip"]
        conjunto = ConjuntoInundaciones(ids, normalizador, aumentar_datos=es_train,
                                        tamano_recorte=tamano_recorte if es_train else None,
                                        semilla=semilla)
        cargadores[particion] = DataLoader(
            conjunto,
            batch_size=tamano_lote if es_train else max(1, tamano_lote // 2),
            shuffle=es_train,
            drop_last=es_train,              # evita un último lote de 1-2 chips (BatchNorm inestable)
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available(),
            worker_init_fn=semilla_por_worker,
            generator=torch.Generator().manual_seed(semilla),
            persistent_workers=num_workers > 0,
        )
        print(f"{particion:8s}: {len(conjunto):3d} chips -> {len(cargadores[particion]):3d} lotes")
    return cargadores


In [ ]:
%%writefile /content/drive/MyDrive/rnap_p2_inundaciones/src/visualizacion_lotes.py
"""
visualizacion_lotes.py
Figuras para inspeccionar lo que REALMENTE entra a la red (después de normalizar y aumentar).
"""
import matplotlib.pyplot as plt
import numpy as np

from .visualizacion import mostrar_mascara


def figura_aumentos(conjunto, indice: int, repeticiones: int = 4, ruta_guardado=None):
    """Muestra el mismo chip con varias transformaciones aleatorias: VH normalizado arriba,
    máscara abajo. Si imagen y máscara están alineadas, el agua (oscura) cae sobre el azul."""
    figura, ejes = plt.subplots(2, repeticiones, figsize=(3 * repeticiones, 6))
    for j in range(repeticiones):
        imagen, etiqueta, id_chip = conjunto[indice]
        ejes[0, j].imshow(imagen[1].numpy(), cmap="gray", vmin=-2.5, vmax=2.5)
        ejes[0, j].set_title(f"VH normalizado #{j + 1}", fontsize=9)
        ejes[0, j].axis("off")
        mostrar_mascara(ejes[1, j], etiqueta.numpy(), "Máscara")
    figura.suptitle(id_chip)
    figura.tight_layout()
    if ruta_guardado:
        figura.savefig(ruta_guardado, dpi=150, bbox_inches="tight")
    return figura


def figura_histograma_normalizado(imagenes, etiquetas, ruta_guardado=None):
    """Histograma de las entradas normalizadas de un lote (debería verse centrado en ~0)."""
    figura, ejes = plt.subplots(1, 2, figsize=(10, 3.5))
    validos = (etiquetas != -1).numpy()
    for b, (eje, nombre) in enumerate(zip(ejes, ["VV", "VH"])):
        valores = imagenes[:, b].numpy()
        for clase, color, texto in [(0, "#b08d57", "No agua"), (1, "#1f5fbf", "Agua")]:
            seleccion = valores[validos & (etiquetas.numpy() == clase)]
            if seleccion.size:
                eje.hist(seleccion, bins=80, range=(-4, 4), density=True, alpha=0.6, color=color, label=texto)
        eje.set_title(f"{nombre} normalizado")
        eje.legend()
        eje.grid(alpha=0.3)
    figura.tight_layout()
    if ruta_guardado:
        figura.savefig(ruta_guardado, dpi=150, bbox_inches="tight")
    return figura


In [ ]:
import time
import numpy as np
import torch

from src.configuracion import *
from src.utilidades import fijar_semilla
from src.descarga import descargar_mano
from src.preprocesamiento import Normalizador
from src.cargadores import crear_cargadores
from src.visualizacion_lotes import figura_aumentos, figura_histograma_normalizado

fijar_semilla(SEMILLA)
descargar_mano()
normalizador = Normalizador.desde_json(DIR_RESULTADOS / 'f1_estadisticas_normalizacion.json')

## 2. Crear los cargadores
- `tamano_lote=8` con recortes de 256×256 en train (en valid/test/bolivia: chips completos 512×512, lote 4).
- La primera vez tarda ~1 min: está cargando los 446 chips en RAM.

In [ ]:
cargadores = crear_cargadores(normalizador, tamano_lote=8, tamano_recorte=256, num_workers=2, semilla=SEMILLA)

## 3. Verificación de un lote por partición
**Qué debe salir:**
- train `[8, 2, 256, 256]`, el resto `[4, 2, 512, 512]`; `float32` para la imagen, `int64` para la etiqueta.
- `NaN: False` siempre.
- Etiqueta solo con valores de `{-1, 0, 1}`.
- Media ≈ 0 y desviación ≈ 1 (aprox.; varía por evento).

In [ ]:
for particion, cargador in cargadores.items():
    imagenes, etiquetas, ids = next(iter(cargador))
    validos = etiquetas != VALOR_SIN_DATO
    print(f"{particion:8s} imagen {tuple(imagenes.shape)} {imagenes.dtype} | etiqueta {tuple(etiquetas.shape)} {etiquetas.dtype}")
    print(f"         NaN: {bool(torch.isnan(imagenes).any())} | valores etiqueta: {sorted(etiquetas.unique().tolist())}"
          f" | media {imagenes.mean():.2f} desv {imagenes.std():.2f}"
          f" | % ignorados {100 * (~validos).float().mean():.1f}% | % agua (válidos) {100 * (etiquetas[validos] == 1).float().mean():.1f}%")

## 4. ¿Imagen y máscara siguen alineadas tras el aumento?
Mismo chip, 4 transformaciones aleatorias. **Lo que valida:** en cada columna, las zonas oscuras de VH deben caer exactamente sobre el azul de la máscara, y cada columna debe estar volteada/rotada distinto.

In [ ]:
conjunto_train = cargadores['train'].dataset
# el chip de train con más agua, para que se note bien
fracciones = [(etq == 1).mean() for _, etq in conjunto_train.cache]
indice = int(np.argmax(fracciones))
figura_aumentos(conjunto_train, indice, repeticiones=4, ruta_guardado=DIR_FIGURAS / 'f2_aumentos.png');

## 5. ¿Cómo quedan las entradas normalizadas?
Mismo histograma de la fase 1 pero ya normalizado: debe estar centrado cerca de 0 y conservar la separación agua / no agua (la normalización **no** debe destruir la información).

In [ ]:
imagenes, etiquetas, _ = next(iter(cargadores['valid']))
figura_histograma_normalizado(imagenes, etiquetas, DIR_FIGURAS / 'f2_histograma_normalizado.png');

## 6. Velocidad del cargador
Si una época de train tarda más de unos segundos, el cuello de botella sería la CPU (y la GPU esperaría). Con caché en RAM debería ser rápido.

In [ ]:
inicio = time.time()
for imagenes, etiquetas, _ in cargadores['train']:
    imagenes = imagenes.to('cuda', non_blocking=True)
print(f"1 época de train ({len(cargadores['train'])} lotes): {time.time() - inicio:.1f} s")

## Para traer a la conversación
La salida de las secciones **3** y **6**, y las figuras de **4** (aumentos) y **5** (histograma normalizado).